# Day 17 - Qdrant Vector Database & Semantic Search

This notebook demonstrates:
- Generating 384-dimensional embeddings
- Creating a Qdrant collection
- Uploading vectors
- Semantic search
- Metadata filtering

**No LLM is used today.**

In [15]:
# !pip install sentence-transformers qdrant-client pandas

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import pandas as pd
import os
from dotenv import load_dotenv

# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("HF_TOKEN"):
    raise ValueError("HF_TOKEN not found. Please check your .env file.")

model=SentenceTransformer("all-MiniLM-L6-v2")
client=QdrantClient(host="localhost",port=6333)
print("Connected")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected


#### Chunks
Assuming, We already have done the chunking

Q: How many vacation days do employees receive?

In [2]:
documents=[
{"department":"HR","text":"Employees receive 18 days of annual leave every year."},
{"department":"HR","text":"Medical leave allows up to 10 days annually."},
{"department":"Travel","text":"International travel requires VP approval."},
{"department":"Travel","text":"Hotel expenses are reimbursed up to company limits."},
{"department":"IT","text":"Company laptops are replaced every four years."},
{"department":"IT","text":"VPN access is mandatory for remote work."},
{"department":"Finance","text":"Expense claims should be submitted within 30 days."},
{"department":"Finance","text":"Salary is credited on the last working day of every month."},
{"department":"Engineering","text":"Python is the preferred language for AI projects."},
{"department":"Engineering","text":"Docker containers simplify deployment."}
]
pd.DataFrame(documents)


,department,text
0,HR,Employees receive 18 days of annual leave ever...
1,HR,Medical leave allows up to 10 days annually.
2,Travel,International travel requires VP approval.
3,Travel,Hotel expenses are reimbursed up to company li...
4,IT,Company laptops are replaced every four years.
5,IT,VPN access is mandatory for remote work.
6,Finance,Expense claims should be submitted within 30 d...
7,Finance,Salary is credited on the last working day of ...
8,Engineering,Python is the preferred language for AI projects.
9,Engineering,Docker containers simplify deployment.


In [3]:
texts=[d["text"] for d in documents]
embeddings=model.encode(texts)
print(embeddings.shape)


(10, 384)


In [16]:
embeddings[0]

array([ 6.49055317e-02,  6.08355887e-02,  6.96704015e-02,  1.78844016e-02,
        4.16478282e-03,  6.06948249e-02, -4.95865606e-02, -8.68155062e-02,
       -2.24852990e-02, -6.53456375e-02,  4.92458269e-02,  3.05002593e-02,
       -7.14240149e-02, -4.38691676e-02, -1.73113178e-02,  2.38916967e-02,
       -5.92406690e-02, -2.23403201e-02,  7.71416677e-03, -9.66793895e-02,
       -1.97028480e-02, -6.46564141e-02, -4.62503470e-02, -7.10236328e-03,
        9.37251281e-03,  2.90031196e-03, -1.61035676e-02, -4.17879224e-02,
       -2.76322719e-02,  6.38967529e-02,  4.20686044e-02,  6.05626851e-02,
        2.73725539e-02, -3.21392864e-02,  4.91284905e-03,  1.69611268e-03,
       -4.38027419e-02, -3.92892817e-03,  1.74630322e-02,  1.94096453e-02,
       -3.93981747e-02,  3.58626060e-02,  8.89555290e-02, -2.66630892e-02,
       -1.14394072e-02, -2.06949245e-02, -8.14028531e-02,  4.05333303e-02,
        6.19838126e-02,  1.35150969e-01,  9.75810736e-02,  4.73400652e-02,
        5.35129122e-02,  

In [17]:
COLLECTION="employee_docs"
try:
    client.delete_collection(COLLECTION)
except:
    pass
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384,distance=Distance.COSINE)
)
points=[]
for i,(doc,vec) in enumerate(zip(documents,embeddings)):
    points.append(PointStruct(id=i,vector=vec.tolist(),payload=doc))
client.upsert(collection_name=COLLECTION,points=points)
print("Uploaded",len(points),"documents. Each doc is a point (position-vector)")


Uploaded 10 documents. Each doc is a point (position-vector)


#### Retrieve first three!

In [18]:
first_three = client.retrieve(collection_name=COLLECTION, ids=[0, 1, 2])
pd.DataFrame(first_three)

,0,1,2,3,4
0,"(id, 0)","(payload, {'department': 'HR', 'text': 'Employ...","(vector, None)","(shard_key, None)","(order_value, None)"
1,"(id, 1)","(payload, {'department': 'HR', 'text': 'Medica...","(vector, None)","(shard_key, None)","(order_value, None)"
2,"(id, 2)","(payload, {'department': 'Travel', 'text': 'In...","(vector, None)","(shard_key, None)","(order_value, None)"


In [19]:
def semantic_search(query,top_k=3):
    q=model.encode(query).tolist()
    results=client.query_points(collection_name=COLLECTION,
                                query=q,limit=top_k).points
    print("\nQuery:",query)
    print("-"*60)
    for r in results:
        print(f"Score: {r.score:.4f}")
        print("Department:",r.payload["department"])
        print(r.payload["text"])
        print()

semantic_search("How many vacation days do employees receive?")
# semantic_search("Who approves international travel?")
# semantic_search("How often are laptops replaced?")



Query: How many vacation days do employees receive?
------------------------------------------------------------
Score: 0.6881
Department: HR
Employees receive 18 days of annual leave every year.

Score: 0.4632
Department: HR
Medical leave allows up to 10 days annually.

Score: 0.4443
Department: Finance
Salary is credited on the last working day of every month.



In [14]:
query="travel approval"
q=model.encode(query).tolist()
results=client.query_points(
    collection_name=COLLECTION,
    query=q,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="department",
                match=MatchValue(value="Travel")
            )
        ]
    ),
    limit=3
).points

for r in results:
    print(r.payload)
    print("Score:",round(r.score,4))
    print()


{'department': 'Travel', 'text': 'International travel requires VP approval.'}
Score: 0.6368

{'department': 'Travel', 'text': 'Hotel expenses are reimbursed up to company limits.'}
Score: 0.2565



## Summary

Today we learned:

- A vector database stores embeddings.
- Each vector is associated with metadata (payload).
- Semantic search retrieves the nearest vectors.
- Metadata filtering narrows the search.

Tomorrow we will connect the retriever to an LLM to build a complete RAG pipeline.
